# STRAINMETER workflow

Notebook operativo per:

1. addestrare `AE1` e `AE2` sui due dataset di training;
2. caricare il file reale in `data/measured`;
3. produrre la timeline con i tre punteggi: type 1, type 2, outlier.

In [ ]:
from pathlib import Path
import sys
import numpy as np

import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

In [ ]:
from src.config import load_config
from src.data import build_dataloaders, load_waveforms
from src.discriminator import run_discriminator
from src.models import Conv1dAutoencoder
from src.plotting import LiveTrainingPlot, plot_reconstructions
from src.train import fit_autoencoder
from src.utils import ensure_directory, resolve_device, set_seed


In [ ]:
config = load_config(PROJECT_ROOT / 'config' / 'config.yaml')
set_seed(config.training.seed)
device = resolve_device(config.training.device)

print('Training datasets:')
for run in config.autoencoders:
    path = (PROJECT_ROOT / run.train_path) if not Path(run.train_path).is_absolute() else Path(run.train_path)
    waveforms = load_waveforms(path, expected_length=config.data.expected_length)
    print(f' - {run.name}: {path} -> {waveforms.shape}')

measured_path = (PROJECT_ROOT / config.discriminator.measured_path) if not Path(config.discriminator.measured_path).is_absolute() else Path(config.discriminator.measured_path)
measured = np.loadtxt(measured_path)
print(f'Measured file: {measured_path} -> {measured.shape}')


In [ ]:
trained_runs = []
training_artifacts = []

for run in config.autoencoders:
    train_path = (PROJECT_ROOT / run.train_path) if not Path(run.train_path).is_absolute() else Path(run.train_path)
    val_path = (PROJECT_ROOT / run.val_path) if run.val_path and not Path(run.val_path).is_absolute() else (Path(run.val_path) if run.val_path else None)

    train_loader, val_loader = build_dataloaders(
        train_path=train_path,
        val_path=val_path,
        expected_length=config.data.expected_length,
        batch_size=config.data.batch_size,
        validation_split=config.data.validation_split,
        shuffle=config.data.shuffle,
        num_workers=config.data.num_workers,
        pin_memory=config.data.pin_memory,
        normalization=config.data.normalization,
        seed=config.training.seed,
    )

    model = Conv1dAutoencoder(
        input_channels=config.model.input_channels,
        waveform_length=config.model.waveform_length,
        encoder_channels=config.model.encoder_channels,
        kernel_size=config.model.kernel_size,
        stride=config.model.stride,
        padding=config.model.padding,
        latent_dim=config.model.latent_dim,
        activation=config.model.activation,
        batch_norm=config.model.batch_norm,
        dropout=config.model.dropout,
        decoder_type=config.model.decoder_type,
        output_activation=config.model.output_activation,
        name=run.name,
    )

    config.model.name = run.name
    output_dir = ensure_directory((PROJECT_ROOT / run.output_dir) if run.output_dir else (PROJECT_ROOT / config.output_dir / run.name.lower()))
    plotter = LiveTrainingPlot(enabled=config.training.live_plot)

    artifact = fit_autoencoder(
        model,
        train_loader,
        val_loader,
        config,
        output_dir=output_dir,
        plotter=plotter if config.training.live_plot else None,
        device=device,
    )
    training_artifacts.append(artifact)
    trained_runs.append((run, output_dir))

    preview_batch = next(iter(val_loader if val_loader is not None else train_loader))
    figure = plot_reconstructions(model.to(device), preview_batch, device=device, max_items=4)
    figure.savefig(output_dir / 'final_reconstructions.png', dpi=150, bbox_inches='tight')
    plt.show()

training_artifacts

In [ ]:
discriminator_artifacts = run_discriminator(
    config,
    trained_runs,
    project_root=PROJECT_ROOT,
    device=device,
)

display(Image(filename=str(discriminator_artifacts.timeline_plot_path)))
discriminator_artifacts